# Source comparison — eLMS vs DataMade

Both sources record the same events: at a council meeting the council votes on an item, and
each alderperson's vote is written down. **eLMS** is the City Clerk's own system; **DataMade**
is a scraped copy. This notebook compares them in small steps:

1. **Date coverage** — which dates does each source have votes on? ← *this notebook, so far*
2. Meetings — per shared date, how many votes each source has.
3. Items — which items are in one source but not the other.
4. Individual votes — does each alder's vote agree (Martin first)?

Sources (downloaded earlier; see README "Data"):
- eLMS raw JSON in `data/raw/elms/` (from `scripts/fetch_elms.py`)
- DataMade nightly SQLite at `data/raw/chicago_council.db`

Kernel: **Chicago Council**.

In [ ]:
from pathlib import Path
import json
import sqlite3

import pandas as pd

# --- Settings (every choice made in this notebook) --------------------------

ELMS_DIR = Path("../data/raw/elms")
DATAMADE_DB_PATH = Path("../data/raw/chicago_council.db")

# eLMS gives each action date as a UTC timestamp. It is converted to a calendar
# date in this timezone so it can be compared with DataMade's plain dates.
# Section 3 shows the timestamps so this conversion can be checked.
LOCAL_TIMEZONE = "America/Chicago"

# Raw date values that are placeholders, not real dates (identified by the user
# 2026-09-30 as entry errors). They are listed in section 5 and do not count as
# coverage. The eLMS value is the raw UTC timestamp — converted to Chicago time
# it would read 1899-12-31.
ELMS_PLACEHOLDER_TIMESTAMPS = ["1900-01-01T00:00:00+00:00"]
DATAMADE_PLACEHOLDER_DATES = ["1900-01-01"]

# Where each source's coverage STARTS.
# eLMS: scripts/fetch_elms.py downloads items active on or after its
#   TERM_START_DATE (since 2026-09-30: every such item with a vote, plus every
#   item on a council agenda since then — see section 17). So eLMS is only
#   complete from that date; its earlier votes are on items that were also
#   active later. They are listed as outside coverage.
# DataMade: its first non-placeholder vote date (computed).
ELMS_COVERAGE_START = "2023-05-15"   # = TERM_START_DATE in scripts/fetch_elms.py
# Where each source's coverage ENDS: its last non-placeholder vote date (computed).

# Compared window (agreed 2026-09-30): only the overlap of the two coverages —
# later of the two starts to earlier of the two ends. Computed in section 6.
# Everything outside it is still shown (section 8), just not compared.

# The per-date chart (section 7) starts here. DataMade goes back to 2011; this
# only keeps the chart readable. Years before it are summarized in section 4.
DATE_CHART_START = "2023-01-01"

# Items are matched on record number alone, whatever their vote dates (agreed
# 2026-09-30). When the two sources give the same item different vote dates, the
# date from this source is the "official" one (used to decide whether the item is
# inside the compared window). Default eLMS because it is the City Clerk's own
# system. The other source's date is used only when this source has no vote.
OFFICIAL_DATE_SOURCE = "eLMS"   # "eLMS" or "DataMade"

# Substitutes (section 19). Decided by the user 2026-09-30: keep eLMS's structure for now.
# - One-to-one substitute: eLMS renames the original record (O... -> SO...), so it is one
#   record with one vote; nothing to merge.
# - Combined substitute: the vote is on the substitute record only; the originals
#   ("Input to Substitute") carry no vote and are NOT given the substitute's vote
#   (DataMade appears to copy it onto them — section 18; not verified in their code).
# TO LOOK INTO LATER: whether an original of a combined substitute should count as voted on.
SUBSTITUTE_RULE = "eLMS structure"

# Section 15 compares titles to find the same item under two different record
# numbers. Titles count as equal after this normalization only: lowercase, and
# runs of whitespace collapsed to one space, trimmed. Nothing fuzzier.
def normalize_title(title):
    return " ".join(str(title or "").lower().split())

# Chart colors: one per source, used the same way in every chart in this notebook.
SOURCE_COLORS = {"eLMS": "#2a78d6", "DataMade": "#eb6834"}   # blue, orange
WINDOW_LINE_COLOR = "#555555"

# Show whole tables rather than truncating them.
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 200)

## 1. eLMS — what one raw item file looks like

eLMS has one JSON file per item ("matter") in `data/raw/elms/matters/`. Below: one example,
the 2025 budget ordinance `SO2024-0013682`. First its top-level fields, then its list of
actions (the history of the item), then the per-member votes attached to one action.

In [ ]:
EXAMPLE_RECORD_NUMBER = "SO2024-0013682"   # 2025 budget ordinance, 2024-12-16

example_matter = None
for matter_path in (ELMS_DIR / "matters").glob("*.json"):
    matter = json.loads(matter_path.read_text())
    if matter["recordNumber"] == EXAMPLE_RECORD_NUMBER:
        example_matter = matter
        print("file:", matter_path.name)
        break

# Top-level fields (lists shown by length only)
top_level_fields = pd.Series({
    field: (f"[list of {len(value)}]" if isinstance(value, list) else value)
    for field, value in example_matter.items()
}, name="value")
display(top_level_fields.to_frame())

In [ ]:
# The item's actions — one row per action. `votes` is shown as a count here.
example_actions = pd.DataFrame(example_matter["actions"])
example_actions["votes"] = example_actions["votes"].apply(lambda votes: len(votes or []))
display(example_actions)

In [ ]:
# Per-member votes for the action(s) that have them.
for action in example_matter["actions"]:
    if action.get("votes"):
        print(f'{action["actionDate"]}  {action["actionName"]}  —  {len(action["votes"])} member votes')
        display(pd.DataFrame(action["votes"]))

## 2. eLMS — the vote table used for comparison

One row per **vote event** = one action (in any item file) that has a non-empty `votes` list.
Built from all item files. Nothing is filtered out.

In [ ]:
elms_vote_event_rows = []
for matter_path in sorted((ELMS_DIR / "matters").glob("*.json")):
    matter = json.loads(matter_path.read_text())
    for action in matter.get("actions") or []:
        member_votes = action.get("votes") or []
        if len(member_votes) == 0:
            continue
        elms_vote_event_rows.append({
            "record_number": matter["recordNumber"],
            "action_date_utc": action["actionDate"],
            "body": action["actionByName"],
            "action_name": action["actionName"],
            "action_text": action["actionText"],
            "meeting_id": action["meetingId"],
            "number_of_member_votes": len(member_votes),
            "history_id": action["historyId"],
        })
elms_vote_events = pd.DataFrame(elms_vote_event_rows)

print(f"eLMS item files read: {len(list((ELMS_DIR / 'matters').glob('*.json'))):,}")
print(f"eLMS vote events:     {len(elms_vote_events):,}")
print("\nColumns:")
display(pd.DataFrame({"column": elms_vote_events.columns,
                      "distinct_values": [elms_vote_events[column].nunique() for column in elms_vote_events.columns]}))
display(elms_vote_events.head(5))

In [ ]:
# Unique values of the short categorical columns, with counts.
for column in ["body", "action_name", "number_of_member_votes"]:
    print(f"--- {column}")
    display(elms_vote_events[column].value_counts(dropna=False).to_frame())

## 3. eLMS — from timestamp to calendar date

`action_date_utc` is a UTC timestamp. Converted to Chicago time, most are midnight (so the
calendar date is unambiguous). The table below counts every time of day that occurs, so the
others are visible. Rows whose Chicago time is **not** midnight are then listed in full.

In [ ]:
action_timestamps = pd.to_datetime(elms_vote_events["action_date_utc"], utc=True)
elms_vote_events["local_timestamp"] = action_timestamps.dt.tz_convert(LOCAL_TIMEZONE)
elms_vote_events["local_date"] = elms_vote_events["local_timestamp"].dt.strftime("%Y-%m-%d")
elms_vote_events["local_time_of_day"] = elms_vote_events["local_timestamp"].dt.strftime("%H:%M")
elms_vote_events["utc_date"] = action_timestamps.dt.strftime("%Y-%m-%d")

display(elms_vote_events.groupby(["local_time_of_day"]).size().rename("vote_events").to_frame())

In [ ]:
not_local_midnight = elms_vote_events[elms_vote_events["local_time_of_day"] != "00:00"]
print(f"Vote events whose Chicago time is not midnight: {len(not_local_midnight):,}")
print(f"...of which the UTC date and Chicago date differ: {(not_local_midnight.utc_date != not_local_midnight.local_date).sum()}")
print("\nCounted by (UTC date, Chicago date, Chicago time):")
display(not_local_midnight.groupby(["utc_date", "local_date", "local_time_of_day"]).size().rename("vote_events").to_frame())

## 4. DataMade — the vote table used for comparison

DataMade stores vote events in the `voteevent` table (one row per vote event), linked to
`bill` (for the record number) and `organization` (the body that voted). All years are loaded.

In [ ]:
datamade = sqlite3.connect(DATAMADE_DB_PATH)

print("voteevent columns:")
display(pd.read_sql("PRAGMA table_info(voteevent)", datamade)[["name", "type"]])

datamade_vote_events = pd.read_sql("""
    SELECT bill.identifier          AS record_number,
           voteevent.start_date     AS local_date,
           organization.name        AS body,
           voteevent.motion_text,
           voteevent.result,
           voteevent.id             AS vote_event_id
    FROM voteevent
    JOIN bill         ON bill.id = voteevent.bill_id
    JOIN organization ON organization.id = voteevent.organization_id
""", datamade)

print(f"DataMade vote events (all years): {len(datamade_vote_events):,}")
print("\nColumns used:")
display(pd.DataFrame({"column": datamade_vote_events.columns,
                      "distinct_values": [datamade_vote_events[column].nunique() for column in datamade_vote_events.columns]}))
display(datamade_vote_events.head(5))

In [ ]:
# The date column: what its values look like (DataMade stores dates as text).
print("Distinct lengths of the start_date text:")
display(datamade_vote_events["local_date"].str.len().value_counts().to_frame())

# Vote events per year, all years.
display(datamade_vote_events["local_date"].str[:4].value_counts().sort_index().rename("vote_events").to_frame())

for column in ["body", "result"]:
    print(f"--- {column}")
    display(datamade_vote_events[column].value_counts(dropna=False).to_frame())

## 5. Placeholder dates

Vote events whose date is one of the placeholder values in settings, listed in full for
both sources. They are set aside from here on (not counted as coverage, not compared).

In [ ]:
elms_is_placeholder = elms_vote_events["action_date_utc"].isin(ELMS_PLACEHOLDER_TIMESTAMPS)
datamade_is_placeholder = datamade_vote_events["local_date"].isin(DATAMADE_PLACEHOLDER_DATES)

print(f"eLMS vote events with a placeholder date: {elms_is_placeholder.sum()}")
display(elms_vote_events[elms_is_placeholder][
    ["action_date_utc", "record_number", "body", "action_name", "action_text", "number_of_member_votes"]])

print(f"DataMade vote events with a placeholder date: {datamade_is_placeholder.sum()}")
display(datamade_vote_events[datamade_is_placeholder][
    ["local_date", "record_number", "body", "motion_text", "result"]])

# The two tables below exclude the placeholder rows.
elms_dated = elms_vote_events[~elms_is_placeholder]
datamade_dated = datamade_vote_events[~datamade_is_placeholder]

## 6. Coverage of each source, and the compared window

Each source's coverage start and end (see settings for how each is defined), then the
window that is compared: the overlap of the two.

In [ ]:
coverage = pd.DataFrame({
    "coverage_start": [ELMS_COVERAGE_START, datamade_dated.local_date.min()],
    "coverage_start_defined_as": ["ELMS_COVERAGE_START setting (download scope)", "first vote date"],
    "coverage_end": [elms_dated.local_date.max(), datamade_dated.local_date.max()],
    "coverage_end_defined_as": ["last vote date", "last vote date"],
    "first_vote_date_in_data": [elms_dated.local_date.min(), datamade_dated.local_date.min()],
}, index=["eLMS", "DataMade"])
display(coverage)

window_start = max(coverage.coverage_start)
window_end = min(coverage.coverage_end)
print(f"Compared window: {window_start} to {window_end}  (both ends inclusive)")

## 7. Vote events per date

One pair of bars per date: eLMS (blue) on the left, DataMade (orange) on the right,
height = number of vote events that source has on that date. Dashed lines = the compared window.
(Dates only a day or two apart will have touching bars.) The same numbers as a table are in the
cell after the chart.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Bar geometry, in days on the date axis: each source's bar is this wide, and is
# shifted this far left (eLMS) or right (DataMade) of its date so the pair sits side by side.
BAR_WIDTH_DAYS = 3
BAR_OFFSET_DAYS = 1.6

elms_events_per_date = elms_dated.groupby("local_date").size().rename("elms_vote_events")
datamade_events_per_date = datamade_dated.groupby("local_date").size().rename("datamade_vote_events")

# One row per date that has votes in either source; 0 where a source has none.
dates_side_by_side = pd.concat([elms_events_per_date, datamade_events_per_date], axis=1).fillna(0).astype(int)
dates_side_by_side.index.name = "local_date"
dates_side_by_side = dates_side_by_side.sort_index()

chart_rows = dates_side_by_side[dates_side_by_side.index >= DATE_CHART_START]
chart_dates = pd.to_datetime(chart_rows.index)

figure, axis = plt.subplots(figsize=(14, 5))
for source, column, offset_days in [("eLMS", "elms_vote_events", -BAR_OFFSET_DAYS),
                                    ("DataMade", "datamade_vote_events", +BAR_OFFSET_DAYS)]:
    axis.bar(chart_dates + pd.Timedelta(days=offset_days), chart_rows[column],
             width=BAR_WIDTH_DAYS, color=SOURCE_COLORS[source], label=source)
for boundary_date in [window_start, window_end]:
    axis.axvline(pd.Timestamp(boundary_date), color=WINDOW_LINE_COLOR, linestyle="--", linewidth=1)
axis.text(pd.Timestamp(window_start), axis.get_ylim()[1], f" window start {window_start}",
          va="top", fontsize=9, color=WINDOW_LINE_COLOR)
axis.text(pd.Timestamp(window_end), axis.get_ylim()[1], f"window end {window_end} ",
          va="top", ha="right", fontsize=9, color=WINDOW_LINE_COLOR)
axis.set_title("Vote events per date, by source", loc="left", fontsize=11)
axis.set_ylabel("vote events")
axis.legend(loc="upper left", bbox_to_anchor=(0, 0.93), frameon=False)
axis.grid(axis="y", color="#e5e5e5", linewidth=0.8)
axis.set_axisbelow(True)
for spine in ["top", "right"]:
    axis.spines[spine].set_visible(False)
axis.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
figure.tight_layout()
plt.show()

In [ ]:
# The chart's numbers as a table (every date from DATE_CHART_START with votes in either source).
display(chart_rows)

## 8. Outside the compared window — noted, not compared

eLMS vote events outside the window are listed in full. DataMade's (which go back to 2011)
are counted by year.

In [ ]:
elms_outside = elms_dated[(elms_dated.local_date < window_start) | (elms_dated.local_date > window_end)]
print(f"eLMS vote events outside the window: {len(elms_outside)}")
display(elms_outside[["local_date", "action_date_utc", "record_number", "body", "action_name", "action_text"]]
        .sort_values("local_date"))

datamade_outside = datamade_dated[(datamade_dated.local_date < window_start) | (datamade_dated.local_date > window_end)]
print(f"DataMade vote events outside the window: {len(datamade_outside):,}")
display(datamade_outside.assign(year=datamade_outside.local_date.str[:4],
                                side=lambda frame: frame.local_date.map(lambda date: "before window" if date < window_start else "after window"))
        .groupby(["side", "year"]).size().rename("vote_events").to_frame())

In [ ]:
# Totals: whole source, placeholders, outside window, inside window.
display(pd.DataFrame({
    "all_vote_events": [len(elms_vote_events), len(datamade_vote_events)],
    "placeholder_date": [elms_is_placeholder.sum(), datamade_is_placeholder.sum()],
    "outside_window": [len(elms_outside), len(datamade_outside)],
    "inside_window": [len(elms_dated) - len(elms_outside), len(datamade_dated) - len(datamade_outside)],
}, index=["eLMS", "DataMade"]))

## 9. What is being counted — one vote, as each source records it

A "vote event" in the counts above means:
- **eLMS**: one entry in an item file's `actions` list that has a non-empty `votes` list,
  dated by the action's `actionDate`. Only item files that were downloaded are counted — and
  `scripts/fetch_elms.py` downloaded an item only if it appeared on the agenda of a downloaded
  full-council meeting.
- **DataMade**: one row of the `voteevent` table, dated by `start_date`. Every row is counted
  (none is dropped by the joins to `bill` and `organization`).

Below: the same item — the 2025 budget ordinance — in both sources, every vote event it has.

In [ ]:
print("eLMS vote events for", EXAMPLE_RECORD_NUMBER)
display(elms_vote_events[elms_vote_events.record_number == EXAMPLE_RECORD_NUMBER][
    ["record_number", "action_date_utc", "local_date", "body", "action_name", "action_text", "number_of_member_votes", "history_id"]])

print("DataMade vote events for", EXAMPLE_RECORD_NUMBER)
datamade_example_events = datamade_vote_events[datamade_vote_events.record_number == EXAMPLE_RECORD_NUMBER]
display(datamade_example_events)

In [ ]:
# The member votes under each of those vote events, counted by value — raw values from each source.
for action in example_matter["actions"]:
    if action.get("votes"):
        print(f"eLMS, action {action['historyId']} ({action['actionDate']}):")
        display(pd.DataFrame(action["votes"])["vote"].value_counts().to_frame())

for vote_event_id in datamade_example_events.vote_event_id:
    print(f"DataMade, voteevent {vote_event_id}:")
    display(pd.read_sql("SELECT option, COUNT(*) AS count FROM personvote WHERE vote_event_id = ? GROUP BY option",
                        datamade, params=[vote_event_id]))

# Step 2 — matching items by record number

Every item has a record number (e.g. `SO2024-0013682`) in both sources: eLMS `recordNumber`,
DataMade `bill.identifier`. Here the two sources are matched on that number **as an exact
string**, and the items that are in only one source are listed row by row.

## 10. Record number formats

Each record number with its digits replaced by `9` and its letters by `A`, counted, with one
example of each. If both sources use the same formats, exact matching is meaningful.
Counted over vote events inside the compared window.

In [ ]:
import re

def record_number_format(record_number):
    """'SO2024-0013682' -> 'A9999-9999999' (letters -> A, digits -> 9)."""
    return re.sub(r"\d", "9", re.sub(r"[A-Za-z]+", "A", record_number))

elms_in_window = elms_dated[(elms_dated.local_date >= window_start) & (elms_dated.local_date <= window_end)]
datamade_in_window = datamade_dated[(datamade_dated.local_date >= window_start) & (datamade_dated.local_date <= window_end)]

for source, frame in [("eLMS", elms_in_window), ("DataMade", datamade_in_window)]:
    print(f"--- {source}")
    display(frame.assign(format=frame.record_number.map(record_number_format))
            .groupby("format").agg(vote_events=("record_number", "size"), example=("record_number", "first")))

## 11. Record numbers — in which source?

Items are matched on **record number alone**; their vote dates don't need to agree.
An item is included if its *official* vote date (from `OFFICIAL_DATE_SOURCE`, or from the
other source when that one has no vote) is inside the window. Its vote dates and vote-event
counts in each source are listed side by side.

In [ ]:
# Per record number, per source: all vote dates (any date, placeholders excluded) and how many vote events.
elms_by_record = elms_dated.groupby("record_number").agg(
    elms_vote_dates=("local_date", lambda dates: ", ".join(sorted(dates))),
    elms_vote_events=("history_id", "size"),
    elms_action_names=("action_name", lambda names: ", ".join(names)),
)
datamade_by_record = datamade_dated.groupby("record_number").agg(
    datamade_vote_dates=("local_date", lambda dates: ", ".join(sorted(dates))),
    datamade_vote_events=("vote_event_id", "size"),
    datamade_results=("result", lambda results: ", ".join(results)),
)

# Every record number with a (non-placeholder) vote in either source, any date.
record_comparison = elms_by_record.join(datamade_by_record, how="outer")
record_comparison.index.name = "record_number"
has_elms_vote = record_comparison.elms_vote_events.notna()
has_datamade_vote = record_comparison.datamade_vote_events.notna()
record_comparison["found_in"] = "both"
record_comparison.loc[has_elms_vote & ~has_datamade_vote, "found_in"] = "eLMS only"
record_comparison.loc[~has_elms_vote & has_datamade_vote, "found_in"] = "DataMade only"

# Official vote date(s): from OFFICIAL_DATE_SOURCE when it has a vote, else from the other source.
official_dates_column, fallback_dates_column = (
    ("elms_vote_dates", "datamade_vote_dates") if OFFICIAL_DATE_SOURCE == "eLMS"
    else ("datamade_vote_dates", "elms_vote_dates"))
record_comparison["official_vote_dates"] = record_comparison[official_dates_column].fillna(
    record_comparison[fallback_dates_column])
record_comparison["official_date_from"] = record_comparison[official_dates_column].notna().map(
    {True: OFFICIAL_DATE_SOURCE, False: "the other source"})

# Keep items with at least one official vote date inside the window.
record_comparison["official_vote_in_window"] = record_comparison.official_vote_dates.map(
    lambda dates: any(window_start <= date <= window_end for date in dates.split(", ")))
print("All record numbers with a vote, by whether their official vote date is in the window:")
display(record_comparison.groupby(["official_vote_in_window", "found_in"]).size().rename("record_numbers").to_frame())
record_comparison = record_comparison[record_comparison.official_vote_in_window]

print(f"Record numbers with an official vote date in the window: {len(record_comparison):,}")
display(record_comparison.found_in.value_counts().to_frame())
display(record_comparison.head(10))

In [ ]:
# Extra context for each record number, from outside the vote tables:
#  - eLMS: was an item file downloaded for this record number? (recordNumber, or its legacyRecordNumber)
#    and its title.
#  - DataMade: does its `bill` table have this record number at all (even with no vote)? and its title.
elms_item_rows = []
for matter_path in sorted((ELMS_DIR / "matters").glob("*.json")):
    matter = json.loads(matter_path.read_text())
    elms_item_rows.append({"record_number": matter["recordNumber"],
                           "matter_id": matter["matterId"],
                           "legacy_record_number": matter.get("legacyRecordNumber") or "",
                           "elms_title": matter.get("title")})   # `shortTitle` is empty in these files
elms_items = pd.DataFrame(elms_item_rows)

datamade_bills = pd.read_sql("SELECT identifier AS record_number, title AS datamade_title FROM bill", datamade)

record_comparison["elms_item_file_downloaded"] = record_comparison.index.isin(elms_items.record_number)
record_comparison["matches_an_elms_legacy_record_number"] = record_comparison.index.isin(
    elms_items.loc[elms_items.legacy_record_number != "", "legacy_record_number"])
record_comparison["in_datamade_bill_table"] = record_comparison.index.isin(datamade_bills.record_number)
record_comparison = (record_comparison
                     .join(elms_items.drop_duplicates("record_number").set_index("record_number")["elms_title"])
                     .join(datamade_bills.drop_duplicates("record_number").set_index("record_number")["datamade_title"]))

def side_by_side(elms_record_numbers, datamade_record_numbers):
    """One row per pair: eLMS record number, vote dates, title | DataMade record number, vote dates, title.
    Either list may contain None (no counterpart). Used wherever two items are compared."""
    elms_record_numbers, datamade_record_numbers = list(elms_record_numbers), list(datamade_record_numbers)
    elms_titles = elms_items.drop_duplicates("record_number").set_index("record_number")["elms_title"]
    datamade_titles = datamade_bills.drop_duplicates("record_number").set_index("record_number")["datamade_title"]
    return pd.DataFrame({
        "elms_record_number": elms_record_numbers,
        "elms_vote_dates": [elms_by_record.elms_vote_dates.get(number) for number in elms_record_numbers],
        "elms_title": [elms_titles.get(number) for number in elms_record_numbers],
        "datamade_record_number": datamade_record_numbers,
        "datamade_vote_dates": [datamade_by_record.datamade_vote_dates.get(number) for number in datamade_record_numbers],
        "datamade_title": [datamade_titles.get(number) for number in datamade_record_numbers],
    })

print("Record numbers per found_in group, with the context columns counted:")
display(record_comparison.groupby("found_in")[
    ["elms_item_file_downloaded", "matches_an_elms_legacy_record_number", "in_datamade_bill_table"]].sum())

## 12. eLMS only — a vote in eLMS, no vote in DataMade on any date

First counted by eLMS vote date (compare with the chart in section 7), then every row.
`in_datamade_bill_table` says whether DataMade knows the item at all (as a bill with no vote).

In [ ]:
elms_only = record_comparison[record_comparison.found_in == "eLMS only"]
print(f"eLMS-only record numbers: {len(elms_only):,}")
display(elms_only.groupby(["elms_vote_dates"]).agg(
    record_numbers=("elms_vote_events", "size"),
    in_datamade_bill_table=("in_datamade_bill_table", "sum")))

In [ ]:
with pd.option_context("display.max_rows", None):
    display(elms_only[["elms_vote_dates", "elms_action_names", "in_datamade_bill_table", "elms_title", "datamade_title"]])

## 13. DataMade only — a vote in DataMade, no vote in eLMS on any date

Same layout. `elms_item_file_downloaded` says whether our eLMS download has an item file for
this record number at all; `matches_an_elms_legacy_record_number` checks eLMS's older numbering.

In [ ]:
datamade_only = record_comparison[record_comparison.found_in == "DataMade only"]
print(f"DataMade-only record numbers: {len(datamade_only):,}")
display(datamade_only.groupby(["datamade_vote_dates"]).agg(
    record_numbers=("datamade_vote_events", "size"),
    elms_item_file_downloaded=("elms_item_file_downloaded", "sum"),
    matches_an_elms_legacy_record_number=("matches_an_elms_legacy_record_number", "sum")))

In [ ]:
with pd.option_context("display.max_rows", None):
    display(datamade_only[["datamade_vote_dates", "datamade_results", "elms_item_file_downloaded",
                           "matches_an_elms_legacy_record_number", "datamade_title"]])

## 14. In both — but the vote dates or the number of vote events differ

These count as **matched** (section 11); `official_vote_dates` is the date used. Listed here
for the record: record numbers where the two sources' vote dates are not identical, or one
source has more vote events than the other. Every row.

In [ ]:
in_both = record_comparison[record_comparison.found_in == "both"]
dates_differ = in_both.elms_vote_dates != in_both.datamade_vote_dates
counts_differ = in_both.elms_vote_events != in_both.datamade_vote_events
print(f"Record numbers in both: {len(in_both):,}")
print(f"  vote dates differ:          {dates_differ.sum():,}")
print(f"  vote-event counts differ:   {counts_differ.sum():,}")
with pd.option_context("display.max_rows", None):
    differing = in_both[dates_differ | counts_differ]
    display(side_by_side(differing.index, differing.index).assign(
        official_vote_dates=differing.official_vote_dates.values,
        elms_vote_events=differing.elms_vote_events.values, datamade_vote_events=differing.datamade_vote_events.values))

## 15. Is a one-sided item really the same item under a different record number?

Two checks on the items from sections 12 and 13:

**15a. Leading zeros / letter case.** Each record number is reduced to
(letters in upper case, year, number as an integer), so `O2023-0001604` and `o2023-1604`
become the same key. Any eLMS-only item whose key equals a DataMade-only item's key is listed.

**15b. Same title.** An item's title often names a specific person, address or permit, so an
identical title under a different record number points to the same item. Titles are compared
after `normalize_title` (settings). Searched in *all* titles the other source has, not only its
one-sided items. `candidates` = how many items in the other source have that exact title
(more than 1 means the title alone can't identify the item).

In [ ]:
def record_number_key(record_number):
    # (letters upper-cased, year, number as integer) — ignores leading zeros and letter case.
    match = re.fullmatch(r"([A-Za-z]+)(\d{4})-(\d+)", record_number.strip())
    if match is None:
        return ("UNPARSED", record_number)
    return (match.group(1).upper(), match.group(2), int(match.group(3)))

one_sided = record_comparison[record_comparison.found_in != "both"].copy()
unparsed = [number for number in one_sided.index if record_number_key(number)[0] == "UNPARSED"]
print(f"One-sided record numbers that don't fit LETTERS+YEAR-NUMBER: {len(unparsed)}", unparsed[:20])

one_sided["record_number_key"] = [record_number_key(number) for number in one_sided.index]
key_pairs = (one_sided[one_sided.found_in == "eLMS only"].reset_index()[["record_number", "record_number_key", "elms_title"]]
             .merge(one_sided[one_sided.found_in == "DataMade only"].reset_index()[["record_number", "record_number_key", "datamade_title"]],
                    on="record_number_key", suffixes=("_elms", "_datamade")))
print(f"15a — eLMS-only / DataMade-only pairs with the same key: {len(key_pairs)}")
display(key_pairs)

In [ ]:
# 15b, direction 1: DataMade-only items — is their DataMade title the title of some eLMS item file?
elms_items["normalized_title"] = elms_items.elms_title.map(normalize_title)
datamade_bills["normalized_title"] = datamade_bills.datamade_title.map(normalize_title)
elms_title_lookup = elms_items.groupby("normalized_title").agg(
    candidates=("record_number", "size"),
    elms_record_numbers=("record_number", lambda numbers: ", ".join(numbers)))
datamade_title_lookup = datamade_bills.groupby("normalized_title").agg(
    candidates=("record_number", "size"),
    datamade_record_numbers=("record_number", lambda numbers: ", ".join(numbers)))

datamade_only_titles = (one_sided[one_sided.found_in == "DataMade only"][["datamade_vote_dates", "datamade_title"]]
                        .assign(normalized_title=lambda frame: frame.datamade_title.map(normalize_title))
                        .join(elms_title_lookup, on="normalized_title"))
found = datamade_only_titles.candidates.notna()
print(f"15b — DataMade-only items: {len(datamade_only_titles):,}; title found among eLMS item files: {found.sum():,}")
with pd.option_context("display.max_rows", None):
    display(datamade_only_titles[found].drop(columns="normalized_title"))

In [ ]:
# 15b, direction 2: eLMS-only items — is their eLMS title the title of some DataMade bill
# under a DIFFERENT record number? (Same record number = section 12's in_datamade_bill_table.)
elms_only_titles = (one_sided[one_sided.found_in == "eLMS only"][["elms_vote_dates", "elms_title"]]
                    .assign(normalized_title=lambda frame: frame.elms_title.map(normalize_title))
                    .join(datamade_title_lookup, on="normalized_title"))
elms_only_titles["title_found_under_another_record_number"] = [
    isinstance(numbers, str) and any(number != record_number for number in numbers.split(", "))
    for record_number, numbers in zip(elms_only_titles.index, elms_only_titles.datamade_record_numbers)]
elsewhere = elms_only_titles.title_found_under_another_record_number
print(f"15b — eLMS-only items: {len(elms_only_titles):,}; title found in DataMade under another record number: {elsewhere.sum():,}")
with pd.option_context("display.max_rows", None):
    display(elms_only_titles[elsewhere].drop(columns=["normalized_title", "title_found_under_another_record_number"]))

In [ ]:
# For the eLMS-only items DataMade DOES have under the same record number (section 12),
# do the two titles agree? A check that the record number points to the same item in both.
same_number = record_comparison[(record_comparison.found_in == "eLMS only") & record_comparison.in_datamade_bill_table]
titles_agree = same_number.elms_title.map(normalize_title) == same_number.datamade_title.map(normalize_title)
print(f"eLMS-only items in DataMade's bill table under the same number: {len(same_number):,}; titles identical: {titles_agree.sum():,}")
print("Rows where the titles differ:")
with pd.option_context("display.max_rows", None):
    display(same_number[~titles_agree][["elms_vote_dates", "elms_title", "datamade_title"]])

## 15c. How do the record numbers differ, when the titles match?

Every (one-sided item, other-source item with the identical title) pair from 15b, one row per
pair, with a label for how the two record numbers relate. The labels are purely mechanical
string comparisons:
- **same number** — the two record numbers are identical (the item exists in both sources;
  one source just has no vote for it).
- **differ only by a leading "S"** — e.g. `O2023-1723` vs `SO2023-1723`.
- **other** — anything else (different year or number).

`candidates` = how many items in the other source share that title. A title shared by many items
(e.g. a generic claims title) can't identify one item on its own — so rows are also split by
whether the title is unique (`candidates == 1`).

In [ ]:
def number_relationship(record_number_a, record_number_b):
    if record_number_a == record_number_b:
        return "same number"
    if record_number_a == "S" + record_number_b or record_number_b == "S" + record_number_a:
        return 'differ only by a leading "S"'
    return "other"

title_pairs = []
for record_number, row in datamade_only_titles[datamade_only_titles.candidates.notna()].iterrows():
    for elms_number in row.elms_record_numbers.split(", "):
        title_pairs.append({"one_sided_in": "DataMade only", "record_number": record_number,
                            "other_source_record_number": elms_number, "candidates": int(row.candidates),
                            "title": row.datamade_title})
for record_number, row in elms_only_titles[elms_only_titles.candidates.notna()].iterrows():
    for datamade_number in row.datamade_record_numbers.split(", "):
        if datamade_number == record_number:
            continue  # same record number = section 12's in_datamade_bill_table, already shown there
        title_pairs.append({"one_sided_in": "eLMS only", "record_number": record_number,
                            "other_source_record_number": datamade_number, "candidates": int(row.candidates),
                            "title": row.elms_title})
title_pairs = pd.DataFrame(title_pairs)
title_pairs["relationship"] = [number_relationship(first, second) for first, second
                               in zip(title_pairs.record_number, title_pairs.other_source_record_number)]
title_pairs["title_is_unique"] = title_pairs.candidates == 1

print("Pairs, by relationship and whether the title is unique:")
display(title_pairs.groupby(["one_sided_in", "relationship", "title_is_unique"]).size().rename("pairs").to_frame())

In [ ]:
# Every pair, grouped by relationship, both sources side by side. Unique titles first.
title_pairs["elms_number"] = title_pairs.record_number.where(title_pairs.one_sided_in == "eLMS only", title_pairs.other_source_record_number)
title_pairs["datamade_number"] = title_pairs.record_number.where(title_pairs.one_sided_in == "DataMade only", title_pairs.other_source_record_number)
with pd.option_context("display.max_rows", None):
    for relationship in ["same number", 'differ only by a leading "S"', "other"]:
        rows = title_pairs[title_pairs.relationship == relationship].sort_values(
            ["title_is_unique", "one_sided_in", "record_number"], ascending=[False, True, True])
        print(f"--- {relationship}: {len(rows)} pairs")
        display(side_by_side(rows.elms_number, rows.datamade_number).assign(
            one_sided_in=rows.one_sided_in.values, candidates=rows.candidates.values))

In [ ]:
# The leading-"S" pattern across ALL one-sided items, not only title matches:
# for each one-sided record number, does the other source have the same number with an "S"
# added or removed? (Any date, vote or not: DataMade's bill table / eLMS item files.)
elms_record_numbers = set(elms_items.record_number)
datamade_record_numbers = set(datamade_bills.record_number)

def s_variant(record_number):
    return record_number[1:] if record_number.startswith("S") else "S" + record_number

s_check = one_sided[["found_in"]].copy()
s_check["s_variant"] = [s_variant(number) for number in s_check.index]
s_check["s_variant_in_other_source"] = [
    (variant in datamade_record_numbers) if found_in == "eLMS only" else (variant in elms_record_numbers)
    for variant, found_in in zip(s_check.s_variant, s_check.found_in)]
s_check["s_variant_has_vote_in_other_source"] = [
    (variant in set(datamade_dated.record_number)) if found_in == "eLMS only" else (variant in set(elms_dated.record_number))
    for variant, found_in in zip(s_check.s_variant, s_check.found_in)]
s_check["direction"] = ["S added" if not number.startswith("S") else "S removed" for number in s_check.index]

print("One-sided items whose S-variant exists in the other source:")
display(s_check.groupby(["found_in", "direction"])[["s_variant_in_other_source", "s_variant_has_vote_in_other_source"]].sum())
s_pairs = s_check[s_check.s_variant_in_other_source]
with pd.option_context("display.max_rows", None):
    display(side_by_side(
        [number if found_in == "eLMS only" else variant for number, variant, found_in in zip(s_pairs.index, s_pairs.s_variant, s_pairs.found_in)],
        [number if found_in == "DataMade only" else variant for number, variant, found_in in zip(s_pairs.index, s_pairs.s_variant, s_pairs.found_in)],
    ).assign(one_sided_in=s_pairs.found_in.values))

# Where DataMade's data comes from

## 16. DataMade's source URLs

DataMade records a source URL for every vote event (table `votesource`). Counted by month and
by the start of the URL, from 2022-06 on.

In [ ]:
vote_sources_by_month = pd.read_sql("""
    SELECT substr(voteevent.start_date, 1, 7)         AS month,
           substr(votesource.url, 1, 45)              AS url_start,
           COUNT(*)                                   AS vote_event_sources
    FROM voteevent
    LEFT JOIN votesource ON votesource.vote_event_id = voteevent.id
    WHERE voteevent.start_date >= '2022-06-01'
    GROUP BY 1, 2
""", datamade)
display(vote_sources_by_month.pivot_table(index="month", columns="url_start", values="vote_event_sources", fill_value=0))
print("One example full URL per note:")
display(pd.read_sql("SELECT note, MIN(url) AS example_url, COUNT(*) AS rows FROM votesource GROUP BY note", datamade))

## 16a. What DataMade's scraper does with eLMS data

DataMade's repository ([datamade/chicago-council-scrapers](https://github.com/datamade/chicago-council-scrapers))
runs the scraper every night, plus a weekly "big scrape", and exports this SQLite file
(`.github/workflows/scrape.yml`). The scraper itself is
[opencivicdata/scrapers-us-municipal `chicago/bills.py`](https://github.com/opencivicdata/scrapers-us-municipal/blob/0f563e49ef77df6bcc40fed7229f42e0f8dae10e/chicago/bills.py)
(last changed 2024-04-27; line numbers refer to that version). What the code does — each
point is checked against our data in the cells that follow:

| # | Code | What it does |
|---|---|---|
| 1 | `scrape.yml`; `bills.py` L64–73, L133 | Asks eLMS `/matter` for items with any action, introduction or record creation in the last **7 days** (nightly) or **180 days** (weekly), then downloads each item's detail. Not agenda-based, unlike our download. |
| 2 | L240–242 | Creates a vote event only if the action has votes **and a non-empty `actionText`**. |
| 3 | L223, L250 | Vote date = `datetime.fromisoformat(actionDate).date()` — the date **as written in the UTC timestamp**, not converted to Chicago time. |
| 4 | L80–131, L155–170, L430 | Substitutes: record numbers with and without a leading "S" (and old-style "legacy" numbers) are merged into **one** bill; the other numbers are kept as alternate identifiers (table `billidentifier`). |
| 5 | L143–150 | Five record numbers are always skipped: O2023-0002065, CL2023-0003775, CL2023-0003676, O2023-0002113, Or2011-189. |
| 6 | L366–370 | Stores the eLMS `matterId` in `bill.extras` as `matter_id`. |
| 7 | L35–41, L261 | Vote values: Yea→yes, Nay→no, Recused→excused, Present→abstain, others lower-cased; **Vacant** votes are dropped. |

**Scrape history** (GitHub Actions, [Scrape workflow runs](https://github.com/datamade/chicago-council-scrapers/actions/workflows/scrape.yml), checked 2026-09-30):
the weekly 180-day scrape (Saturdays) has **failed every week from at least 2025-10-04 through 2026-09-26**
(also 2025-08-16), with `KeyError: 'Submitted'` at `bills.py` line 224: eLMS uses an action name the
scraper's `ACTION` table doesn't list, and the whole run stops. Nightly 7-day scrapes succeeded.
So since then, DataMade only picks up an item if the nightly scrape sees it within 7 days of its
latest action (point 1).

In [ ]:
# Check of point 2: eLMS vote events in the window with an EMPTY action_text, by where the item was found.
elms_in_window_with_group = elms_in_window.join(record_comparison["found_in"], on="record_number")
elms_in_window_with_group["action_text_is_empty"] = elms_in_window_with_group.action_text.fillna("").str.strip() == ""
display(pd.crosstab(elms_in_window_with_group.found_in, elms_in_window_with_group.action_text_is_empty, margins=True))

In [ ]:
# Check of point 3: for items in both sources with different vote dates (section 14), the eLMS
# timestamp, its UTC date, its Chicago date, and DataMade's date.
differing_numbers = in_both[dates_differ].index
with pd.option_context("display.max_rows", None):
    display(elms_dated[elms_dated.record_number.isin(differing_numbers)][
        ["record_number", "action_date_utc", "utc_date", "local_date"]]
        .merge(datamade_dated[["record_number", "local_date"]].rename(columns={"local_date": "datamade_date"}),
               on="record_number", how="left")
        .assign(datamade_equals_utc_date=lambda frame: frame.datamade_date == frame.utc_date,
                datamade_equals_chicago_date=lambda frame: frame.datamade_date == frame.local_date)
        .sort_values("record_number"))

In [ ]:
# Checks of points 4 and 6: DataMade's alternate identifiers and stored eLMS matter_id.
datamade_bill_ids = pd.read_sql("""
    SELECT bill.identifier AS record_number,
           json_extract(bill.extras, '$.matter_id') AS matter_id,
           group_concat(billidentifier.identifier, ', ') AS alternate_identifiers
    FROM bill
    LEFT JOIN billidentifier ON billidentifier.bill_id = bill.id
    GROUP BY bill.id
""", datamade)
print("DataMade bills, and how many have a matter_id / alternate identifiers:")
display(pd.DataFrame({
    "bills": [len(datamade_bill_ids)],
    "with_matter_id": [datamade_bill_ids.matter_id.notna().sum()],
    "with_alternate_identifiers": [datamade_bill_ids.alternate_identifiers.notna().sum()],
}))
print("Examples of bills with alternate identifiers:")
display(datamade_bill_ids[datamade_bill_ids.alternate_identifiers.notna() & datamade_bill_ids.matter_id.notna()].head(10))

## 16b. Matching on eLMS's own item id (`matterId`) instead of the record number

DataMade stores each bill's eLMS `matterId` (point 6). Here each one-sided item from sections
12–13 is looked up by its matterId in the other source. Rows are shown side by side.
- **DataMade-only** items: DataMade's `matter_id` → do we have an eLMS item file with that
  `matterId`, and under what record number?
- **eLMS-only** items: our `matterId` → does DataMade have a bill with that `matter_id`,
  under what record number, and does that bill have a vote?

In [ ]:
datamade_by_matter_id = datamade_bill_ids.dropna(subset=["matter_id"]).drop_duplicates("matter_id").set_index("matter_id")
elms_by_matter_id = elms_items.drop_duplicates("matter_id").set_index("matter_id")

# DataMade-only items -> eLMS item file with the same matterId?
datamade_only_lookup = pd.DataFrame({"datamade_record_number": datamade_only.index})
datamade_only_lookup["matter_id"] = datamade_only_lookup.datamade_record_number.map(
    datamade_bill_ids.drop_duplicates("record_number").set_index("record_number").matter_id)
datamade_only_lookup["elms_record_number_for_matter_id"] = datamade_only_lookup.matter_id.map(elms_by_matter_id.record_number)
datamade_only_lookup["outcome"] = [
    "no matter_id in DataMade" if pd.isna(matter_id)
    else ("matterId not in our eLMS download" if pd.isna(elms_number)
          else ("same record number in eLMS" if elms_number == datamade_number else "DIFFERENT record number in eLMS"))
    for matter_id, elms_number, datamade_number in zip(datamade_only_lookup.matter_id,
                                                       datamade_only_lookup.elms_record_number_for_matter_id,
                                                       datamade_only_lookup.datamade_record_number)]
print("DataMade-only items, looked up by matterId in our eLMS download:")
display(datamade_only_lookup.outcome.value_counts().to_frame())

found_in_elms = datamade_only_lookup[datamade_only_lookup.elms_record_number_for_matter_id.notna()]
print("Rows where our eLMS download has the matterId:")
with pd.option_context("display.max_rows", None):
    display(side_by_side(found_in_elms.elms_record_number_for_matter_id, found_in_elms.datamade_record_number)
            .assign(outcome=found_in_elms.outcome.values))

In [ ]:
# eLMS-only items -> DataMade bill with the same matter_id?
elms_only_lookup = pd.DataFrame({"elms_record_number": elms_only.index})
elms_only_lookup["matter_id"] = elms_only_lookup.elms_record_number.map(
    elms_items.drop_duplicates("record_number").set_index("record_number").matter_id)
elms_only_lookup["datamade_record_number_for_matter_id"] = elms_only_lookup.matter_id.map(datamade_by_matter_id.record_number)
elms_only_lookup["datamade_bill_has_vote"] = elms_only_lookup.datamade_record_number_for_matter_id.isin(datamade_dated.record_number)
elms_only_lookup["outcome"] = [
    "matterId not in DataMade" if pd.isna(datamade_number)
    else ("same record number in DataMade" if datamade_number == elms_number else "DIFFERENT record number in DataMade")
    for datamade_number, elms_number in zip(elms_only_lookup.datamade_record_number_for_matter_id, elms_only_lookup.elms_record_number)]
print("eLMS-only items, looked up by matterId in DataMade:")
display(elms_only_lookup.groupby(["outcome", "datamade_bill_has_vote"]).size().rename("items").to_frame())

different = elms_only_lookup[elms_only_lookup.outcome == "DIFFERENT record number in DataMade"]
print("Rows where DataMade has the same matterId under a different record number:")
with pd.option_context("display.max_rows", None):
    display(side_by_side(different.elms_record_number, different.datamade_record_number_for_matter_id)
            .assign(datamade_bill_has_vote=different.datamade_bill_has_vote.values))

## 17. The eLMS download after the 2026-09-30 fix

Sections 13 and 16b showed items DataMade has that our download did not. Cause: the first
download found items only through meeting agendas, and agendas are incomplete (21 council
meetings have an empty agenda in the API; items reported back from committee can be voted
without being on the agenda). `scripts/fetch_elms.py` now also takes items from the `/matter`
list, filtered to **matters active in the term that have at least one action with votes**
(`MATTER_LIST_FILTER` in the script). The list is fetched in date windows instead of pages,
because page order is not stable (see the script's docstring).

This section checks the result:
- **17a** the list step's own checks, as saved in `matter_list.json`;
- **17b** whether the "has votes" filter agrees with the votes in each downloaded item file;
- **17c** whether every matterId DataMade has a vote for is now downloaded.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
import fetch_elms   # only for its constants (filters, KNOWN_DELETED_MATTER_IDS); nothing is fetched here

matter_list_file = json.loads((ELMS_DIR / "matter_list.json").read_text())
print("retrieved_at:", matter_list_file["retrieved_at"])
print("filter:", matter_list_file["filter"])
print("windows planned from:", matter_list_file["window_planning_source"])
print("problems recorded by the checks:", matter_list_file["problems"] or "none")

matter_list = pd.DataFrame(matter_list_file["data"])
windows = pd.DataFrame(matter_list_file["windows"])
display(pd.DataFrame({
    "windows": [len(windows)],
    "windows where rows_returned != count": [(windows.rows_returned != windows["count"]).sum()],
    "rows in all windows": [windows.rows_returned.sum()],
    "unique matterIds": [matter_list.matterId.nunique()],
    "whole-query count reported by the API": [matter_list_file["whole_query_count"]],
}).T.rename(columns={0: "value"}))
print("First and last windows:")
display(pd.concat([windows.head(3), windows.tail(3)]))

In [ ]:
# 17b. For every downloaded item file: does it have votes, and is it in the has-votes list?
listed_matter_ids = set(matter_list.matterId)
item_file_rows = []
for matter_path in sorted((ELMS_DIR / "matters").glob("*.json")):
    matter = json.loads(matter_path.read_text())
    item_file_rows.append({
        "matter_id": matter["matterId"],
        "record_number": matter["recordNumber"],
        "has_votes_in_file": any(action.get("votes") for action in matter.get("actions") or []),
        "in_has_votes_list": matter["matterId"] in listed_matter_ids,
        "file_lastPublicationDate": matter.get("lastPublicationDate"),
        "file_saved": pd.Timestamp(matter_path.stat().st_mtime, unit="s").strftime("%Y-%m-%d"),
    })
item_files = pd.DataFrame(item_file_rows)
print(f"Item files in matters/: {len(item_files):,}")
display(pd.crosstab(item_files.has_votes_in_file, item_files.in_has_votes_list, margins=True))

filter_disagrees = item_files[item_files.has_votes_in_file != item_files.in_has_votes_list].copy()
filter_disagrees["list_lastPublicationDate"] = filter_disagrees.matter_id.map(
    matter_list.set_index("matterId").lastPublicationDate)
# A matter not in the has-votes list has no list row; take its date from the all-matters list instead.
all_active_list = pd.DataFrame(json.loads((ELMS_DIR / "matter_list_all_active_in_term.json").read_text())["data"])
filter_disagrees["all_active_list_lastPublicationDate"] = filter_disagrees.matter_id.map(
    all_active_list.drop_duplicates("matterId").set_index("matterId").lastPublicationDate)
print("Files where the filter and the file disagree:")
display(filter_disagrees)

In [ ]:
# 17c. Every matterId DataMade has a vote for in the compared window — where is it now?
datamade_votes_in_window = (datamade_dated[(datamade_dated.local_date >= window_start) & (datamade_dated.local_date <= window_end)]
                            .merge(datamade_bill_ids[["record_number", "matter_id"]].drop_duplicates("record_number"),
                                   on="record_number", how="left"))
datamade_matter_ids = set(datamade_votes_in_window.matter_id.dropna())
check_folder_ids = {path.stem for path in (ELMS_DIR / "matters_datamade_check").glob("*.json")}
known_deleted_ids = set(fetch_elms.KNOWN_DELETED_MATTER_IDS)

def where_is(matter_id):
    if matter_id in set(item_files.matter_id): return "matters/ (main download)"
    if matter_id in check_folder_ids: return "matters_datamade_check/ (section 18)"
    if matter_id in known_deleted_ids: return "known deleted from eLMS (HTTP 404)"
    return "NOT DOWNLOADED"

print(f"DataMade matterIds with a vote in the compared window: {len(datamade_matter_ids):,}")
print(f"DataMade vote events in the window with no matter_id: {datamade_votes_in_window.matter_id.isna().sum()}")
display(pd.Series([where_is(matter_id) for matter_id in datamade_matter_ids]).value_counts().rename("matterIds").to_frame())
print("Known deleted (fetch_elms.KNOWN_DELETED_MATTER_IDS):")
display(pd.DataFrame([(matter_id, number, date) for matter_id, (number, date) in fetch_elms.KNOWN_DELETED_MATTER_IDS.items()],
                     columns=["matter_id", "datamade_record_number", "datamade_vote_date"]))

## 18. Votes that DataMade has but eLMS's item records do not

Some matterIds DataMade has a vote for are in eLMS, but their eLMS item record (`/matter/{id}`)
has **no action with votes**, so the has-votes list leaves them out. They are the DataMade
matterIds in the window that are not in `matter_list.json`, minus the known-deleted ones.
Those not already in `matters/` were fetched once into `matters_datamade_check/`.

- **18a** the items, side by side;
- **18b** an item whose saved file had votes but whose record changed since (section 17b);
- **18c** whether the votes still exist on the *meeting* side of eLMS,
  `/meeting/{meetingId}/matter/{matterId}/votes`, with control pairs that should come back empty.

18b and 18c write request files to `data/raw/elms/check_requests/`; the fetches are run with
`scripts/fetch_elms.py` (commands printed below), then this notebook is re-run to show them.

In [ ]:
# --- Settings for section 18 ---
CHECK_REQUESTS_DIR = ELMS_DIR / "check_requests"
# 18c controls: for each meeting checked, this many matters that were CREATED AFTER the meeting
# (so cannot have been voted at it) are asked about too. Their answers should be empty.
CONTROL_MATTERS_PER_MEETING = 1

# 18a. The items: DataMade vote in the window, matterId in eLMS, not in the has-votes list.
no_vote_in_elms_ids = sorted(datamade_matter_ids - listed_matter_ids - known_deleted_ids)
(CHECK_REQUESTS_DIR / "datamade_only_matter_ids.json").write_text(json.dumps({
    **json.loads((CHECK_REQUESTS_DIR / "datamade_only_matter_ids.json").read_text()),
    "matter_ids": sorted(set(no_vote_in_elms_ids) - set(item_files.matter_id))}, indent=1))

datamade_dates_by_id = datamade_votes_in_window.groupby("matter_id").agg(
    datamade_record_number=("record_number", "first"),
    datamade_vote_dates=("local_date", lambda dates: ", ".join(sorted(set(dates)))))
rows = []
for matter_id in no_vote_in_elms_ids:
    folder = "matters" if matter_id in set(item_files.matter_id) else "matters_datamade_check"
    file_path = ELMS_DIR / folder / f"{matter_id}.json"
    if not file_path.exists():
        rows.append({"matter_id": matter_id, "folder": "NOT FETCHED YET"}); continue
    matter = json.loads(file_path.read_text())
    rows.append({
        "matter_id": matter_id, "folder": folder,
        "datamade_record_number": datamade_dates_by_id.datamade_record_number.get(matter_id),
        "datamade_vote_dates": datamade_dates_by_id.datamade_vote_dates.get(matter_id),
        "elms_record_number": matter["recordNumber"],
        "elms_status": matter.get("status"), "elms_subStatus": matter.get("subStatus"),
        "elms_lastPublicationDate": (matter.get("lastPublicationDate") or "")[:10],
        "elms_actions_with_votes": sum(bool(action.get("votes")) for action in matter.get("actions") or []),
        "elms_actions (date name)": " | ".join(f"{(action.get('actionDate') or '')[:10]} {action.get('actionName')}"
                                              for action in sorted(matter.get("actions") or [], key=lambda action: action.get("actionDate") or "")),
        "elms_title": (matter.get("title") or "")[:100],
    })
no_vote_in_elms = pd.DataFrame(rows)
print(f"Items: {len(no_vote_in_elms)}")
display(no_vote_in_elms.folder.value_counts().to_frame())
print("By eLMS status, and by the month the eLMS record was last published:")
display(no_vote_in_elms.groupby(["elms_status", "elms_subStatus"]).size().rename("items").to_frame())
display(no_vote_in_elms.elms_lastPublicationDate.str[:7].value_counts().sort_index().rename("items").to_frame())
with pd.option_context("display.max_rows", None):
    display(no_vote_in_elms.drop(columns=["matter_id"]))

In [ ]:
# 18b. Items whose saved file has votes but which are not in the has-votes list (from 17b):
# re-fetch them into matters_recheck/ and compare the actions.
recheck_ids = sorted(filter_disagrees[filter_disagrees.has_votes_in_file].matter_id)
(CHECK_REQUESTS_DIR / "recheck_matter_ids.json").write_text(json.dumps({
    "why": "saved file in matters/ has votes, but the matter is not in matter_list.json (section 17b)",
    "matter_ids": recheck_ids}, indent=1))
print(f"{len(recheck_ids)} to re-check. Fetch with:")
print("  python scripts/fetch_elms.py matters_by_id data/raw/elms/check_requests/recheck_matter_ids.json matters_recheck")

def action_table(matter):
    return pd.DataFrame([{"date": (action.get("actionDate") or "")[:10], "action": action.get("actionName"),
                          "by": action.get("actionByName"), "votes": len(action.get("votes") or []),
                          "historyId": action.get("historyId")} for action in matter.get("actions") or []]
                        ).sort_values("date")

for matter_id in recheck_ids:
    saved = json.loads((ELMS_DIR / "matters" / f"{matter_id}.json").read_text())
    recheck_path = ELMS_DIR / "matters_recheck" / f"{matter_id}.json"
    print(f"\n=== {saved['recordNumber']}: {saved['title'][:90]}")
    print(f"saved copy (file date {item_files.set_index('matter_id').file_saved[matter_id]}):")
    display(action_table(saved))
    if not recheck_path.exists():
        print("re-check not fetched yet"); continue
    recheck = json.loads(recheck_path.read_text())
    print(f"re-check (file date {pd.Timestamp(recheck_path.stat().st_mtime, unit='s'):%Y-%m-%d}):")
    display(action_table(recheck))
    display(pd.DataFrame({field: [saved.get(field), recheck.get(field)]
                          for field in ["status", "subStatus", "finalActionDate", "controllingBody", "lastPublicationDate"]},
                         index=["saved", "re-check"]).T)

In [ ]:
# 18c. Meeting-side votes. Pairs = each item from 18a/18b x each council meeting on its DataMade
# vote date(s), plus control pairs (matters created after the meeting).
council_meetings = pd.DataFrame(json.loads((ELMS_DIR / "meeting_list.json").read_text())["data"])
council_meetings = council_meetings[council_meetings.body == fetch_elms.COUNCIL_BODY_NAME]
council_meetings["local_date"] = (pd.to_datetime(council_meetings.date, utc=True)
                                  .dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%Y-%m-%d"))

item_ids_to_check = sorted(set(no_vote_in_elms_ids) | set(recheck_ids))
pairs = []
for matter_id in item_ids_to_check:
    for vote_date in datamade_dates_by_id.datamade_vote_dates.get(matter_id, "").split(", "):
        for meeting_id in council_meetings[council_meetings.local_date == vote_date].meetingId:
            pairs.append({"meeting_id": meeting_id, "matter_id": matter_id, "kind": "item"})
matter_list_by_created = matter_list.sort_values("recordCreateDate")
for meeting_id in sorted({pair["meeting_id"] for pair in pairs}):
    meeting_date = council_meetings.set_index("meetingId").date[meeting_id]
    created_after = matter_list_by_created[matter_list_by_created.recordCreateDate > meeting_date]
    for matter_id in created_after.matterId.head(CONTROL_MATTERS_PER_MEETING):
        pairs.append({"meeting_id": meeting_id, "matter_id": matter_id, "kind": "control"})
(CHECK_REQUESTS_DIR / "meeting_matter_votes_pairs.json").write_text(json.dumps({"pairs": pairs}, indent=1))
print(f"{len(pairs)} pairs ({sum(p['kind'] == 'item' for p in pairs)} item, {sum(p['kind'] == 'control' for p in pairs)} control). Fetch with:")
print("  python scripts/fetch_elms.py meeting_matter_votes data/raw/elms/check_requests/meeting_matter_votes_pairs.json")

record_number_by_id = {**all_active_list.drop_duplicates("matterId").set_index("matterId").recordNumber.to_dict(),
                       **matter_list.set_index("matterId").recordNumber.to_dict()}
result_rows = []
for pair in pairs:
    result_path = ELMS_DIR / "meeting_matter_votes" / f"{pair['meeting_id']}_{pair['matter_id']}.json"
    votes = json.loads(result_path.read_text()) if result_path.exists() else None
    tally = pd.Series([vote.get("vote") for vote in votes]).value_counts().to_dict() if votes else {}
    result_rows.append({"kind": pair["kind"], "record_number": record_number_by_id.get(pair["matter_id"]),
                        "meeting_date": council_meetings.set_index("meetingId").date[pair["meeting_id"]][:16],
                        "fetched": votes is not None, "member_votes": len(votes) if votes else 0,
                        "tally": ", ".join(f"{name} {count}" for name, count in tally.items())})
meeting_votes = pd.DataFrame(result_rows)
print("Summary (member_votes > 0 means the meeting side has a roll call for the pair):")
display(meeting_votes.assign(has_roll_call=meeting_votes.member_votes > 0)
        .groupby(["kind", "fetched", "has_roll_call"]).size().rename("pairs").to_frame())
with pd.option_context("display.max_rows", None):
    display(meeting_votes.sort_values(["kind", "meeting_date", "record_number"]))

### 18d. The "Input to Substitute" items

Most items in 18a have eLMS subStatus *Input to Substitute*. Below, for each item in 18a:
does a record numbered `"S" + its number` exist in our download, and does that substitute
have a vote on DataMade's vote date? (Pure lookups, no interpretation.) Then the 18c
meeting-side results, counted by subStatus.

In [ ]:
elms_votes_by_record = elms_dated.groupby("record_number").local_date.apply(lambda dates: set(dates))
downloaded_record_numbers = set(item_files.record_number)

substitute_lookup = no_vote_in_elms[["datamade_record_number", "datamade_vote_dates", "elms_record_number",
                                     "elms_status", "elms_subStatus"]].copy()
substitute_lookup["substitute_record_number"] = "S" + substitute_lookup.elms_record_number
substitute_lookup["substitute_downloaded"] = substitute_lookup.substitute_record_number.isin(downloaded_record_numbers)
substitute_lookup["substitute_elms_vote_dates"] = substitute_lookup.substitute_record_number.map(
    lambda number: ", ".join(sorted(elms_votes_by_record.get(number, set()))))
substitute_lookup["substitute_voted_on_datamade_date"] = [
    bool(set(datamade_dates.split(", ")) & elms_votes_by_record.get(number, set()))
    for datamade_dates, number in zip(substitute_lookup.datamade_vote_dates, substitute_lookup.substitute_record_number)]
display(substitute_lookup.groupby(["elms_subStatus", "substitute_downloaded", "substitute_voted_on_datamade_date"])
        .size().rename("items").to_frame())
with pd.option_context("display.max_rows", None):
    display(substitute_lookup.sort_values(["elms_subStatus", "datamade_record_number"]))

# Same lookup, following the item's own pointers instead of the "S" prefix:
# `supersededBy` and `relations` (record numbers) in the item file from 18a.
def item_file(matter_id):
    for folder in ["matters", "matters_datamade_check"]:
        path = ELMS_DIR / folder / f"{matter_id}.json"
        if path.exists():
            return json.loads(path.read_text())

def as_record_numbers(value):
    """supersededBy / relations can be a string, a list of strings, or a list of dicts — show what is there."""
    if not value: return []
    if isinstance(value, str): return [value.strip()]
    return [(entry.get("recordNumber") if isinstance(entry, dict) else str(entry)).strip() for entry in value]

pointer_rows = []
for row in no_vote_in_elms.itertuples():
    matter = item_file(row.matter_id)
    for pointer_field in ["supersededBy", "relations"]:
        for number in as_record_numbers(matter.get(pointer_field)):
            pointer_rows.append({
                "elms_record_number": row.elms_record_number, "elms_subStatus": row.elms_subStatus,
                "datamade_vote_dates": row.datamade_vote_dates, "pointer_field": pointer_field,
                "points_to": number, "points_to_downloaded": number in downloaded_record_numbers,
                "points_to_elms_vote_dates": ", ".join(sorted(elms_votes_by_record.get(number, set()))),
                "points_to_voted_on_datamade_date": bool(set(row.datamade_vote_dates.split(", "))
                                                         & elms_votes_by_record.get(number, set())),
            })
pointers = pd.DataFrame(pointer_rows)
print("Raw supersededBy / relations values of the first 3 items (to show their shape):")
for row in no_vote_in_elms.head(3).itertuples():
    matter = item_file(row.matter_id)
    print(f"  {row.elms_record_number}: supersededBy={matter.get('supersededBy')!r}  relations={matter.get('relations')!r}")
print(f"\nItems with at least one pointer: {pointers.elms_record_number.nunique()} of {len(no_vote_in_elms)}")
display(pointers.groupby(["elms_subStatus", "pointer_field", "points_to_downloaded", "points_to_voted_on_datamade_date"])
        .size().rename("pointers").to_frame())
with pd.option_context("display.max_rows", None):
    display(pointers.sort_values(["elms_subStatus", "elms_record_number"]))

# 18c results by subStatus (item pairs only).
substatus_by_number = no_vote_in_elms.set_index("elms_record_number").elms_subStatus.to_dict()
substatus_by_number.update({number: "(18b re-check)" for number in
                            filter_disagrees[filter_disagrees.has_votes_in_file].record_number})
item_pairs = meeting_votes[meeting_votes.kind == "item"].assign(
    elms_subStatus=lambda frame: frame.record_number.map(substatus_by_number),
    has_roll_call=lambda frame: frame.member_votes > 0)
print("Meeting-side roll call (18c), by eLMS subStatus of the item:")
display(pd.crosstab(item_pairs.elms_subStatus, item_pairs.has_roll_call, margins=True))

## 19. What a substitute looks like in eLMS

A *substitute* is a replacement version of an item, usually adopted in committee; the council
then votes on the substitute. eLMS files it as its **own record**, normally the original's
number with an `S` prefix. (That description is general knowledge — the cells below show what
the data actually contains.)

- **19a** how substitutes in our download link back to their originals (which fields);
- **19b** three examples, substitute and original(s) with their full action histories:
  the 2025 budget, a one-to-one zoning substitute, and a substitute combining many originals.

Originals are mostly not downloaded (they usually have no vote). The cell writes the ones
it needs to a request file; fetch them with the printed command, then re-run.

**Decision (2026-09-30): `SUBSTITUTE_RULE = "eLMS structure"`** (settings cell): votes stay
where eLMS puts them, on the substitute; originals of a combined substitute get no vote.
Marked to look into in more detail later.

In [ ]:
# --- Settings for section 19 ---
SUBSTITUTE_EXAMPLES = [
    "SO2024-0013682",   # 2025 budget (Annual Appropriation Ordinance)
    "SO2024-0013057",   # zoning reclassification, 2411-2423 N Kedzie — one original
    "SO2023-0004139",   # "Parking prohibited at all times (except for disabled)" — 63 originals
]
ORIGINALS_SHOWN_PER_EXAMPLE = 3   # a combined substitute lists many originals; show only the first few
# Control for the record-number lookup: a number that certainly exists (the budget substitute).
RECORD_NUMBER_LOOKUP_CONTROL = "SO2024-0013682"

# 19a. Every downloaded record whose number starts with "S": what its link fields contain.
substitute_rows = []
for matter_path in sorted((ELMS_DIR / "matters").glob("*.json")):
    matter = json.loads(matter_path.read_text())
    number = matter["recordNumber"]
    if not number.startswith("S"):
        continue
    relations = [str(relation).strip() for relation in matter.get("relations") or []]
    substitute_rows.append({
        "record_number": number,
        "originalRecordNumber": matter.get("originalRecordNumber") or "",
        "originalRecordNumber_is_number_without_S": (matter.get("originalRecordNumber") or "") == number[1:],
        "relations_count": len(relations),
        "relations_is_only_itself": relations == [number],
        "subStatus": matter.get("subStatus"),
    })
substitutes = pd.DataFrame(substitute_rows)
print(f"Downloaded records whose number starts with 'S': {len(substitutes):,}")
display(substitutes.groupby(["originalRecordNumber_is_number_without_S", "relations_is_only_itself"])
        .size().rename("records").to_frame())
print("relations_count, when relations is not just the record itself:")
display(substitutes[~substitutes.relations_is_only_itself].relations_count.value_counts().sort_index().rename("records").to_frame())

In [ ]:
# 19b. The examples. Originals = originalRecordNumber, plus relations other than the record itself.
matter_id_by_number = all_active_list.drop_duplicates("recordNumber").set_index("recordNumber").matterId.to_dict()
matter_id_by_number.update(item_files.set_index("record_number").matter_id.to_dict())
example_folders = ["matters", "matters_datamade_check", "matters_substitute_examples"]

def find_item_file(record_number):
    matter_id = matter_id_by_number.get(record_number)
    for folder in example_folders:
        path = ELMS_DIR / folder / f"{matter_id}.json"
        if matter_id and path.exists():
            return json.loads(path.read_text())

def originals_of(substitute):
    numbers = [substitute.get("originalRecordNumber") or ""] + [str(r).strip() for r in substitute.get("relations") or []]
    return [n for n in dict.fromkeys(numbers) if n and n != substitute["recordNumber"]]

def lookup_result(record_number):
    """What /matter/recordNumber/{n} returned, from the matters_by_id output folder and its _fetch_log.json."""
    folder = ELMS_DIR / "matters_substitute_examples"
    if (folder / f"recordNumber_{record_number}.json").exists():
        return "found"
    log_path = folder / "_fetch_log.json"
    if log_path.exists():
        for failure in json.loads(log_path.read_text())["failures"]:
            if failure["api_path"].endswith("/" + record_number):
                return f"{failure['reason']} (fetch ran {json.loads(log_path.read_text())['ran_at'][:10]})"
    return "not fetched yet"

KEY_FIELDS = ["recordNumber", "originalRecordNumber", "status", "subStatus", "introductionDate", "finalActionDate",
              "controllingBody", "title"]
to_fetch = []
for example_number in SUBSTITUTE_EXAMPLES:
    substitute = find_item_file(example_number)
    originals = originals_of(substitute)
    print(f"\n==================== {example_number}: {len(originals)} original(s): {', '.join(originals[:10])}{' ...' if len(originals) > 10 else ''}")
    shown = [substitute] + [find_item_file(number) or {"recordNumber": number} for number in originals[:ORIGINALS_SHOWN_PER_EXAMPLE]]
    display(pd.DataFrame({record["recordNumber"]: {field: str(record.get(field, "(not downloaded)"))[:90] for field in KEY_FIELDS}
                          for record in shown}))
    for record in shown:
        if "actions" not in record:
            to_fetch.append(record["recordNumber"])
            lookup_path = ELMS_DIR / "matters_substitute_examples" / f"recordNumber_{record['recordNumber']}.json"
            in_all_active_list = record["recordNumber"] in set(all_active_list.recordNumber)
            print(f"--- {record['recordNumber']}: in the all-active list: {in_all_active_list}; "
                  f"lookup by record number: {lookup_result(record['recordNumber'])}")
            if lookup_path.exists():
                looked_up = json.loads(lookup_path.read_text())
                print(f"    returned recordNumber={looked_up.get('recordNumber')!r}, matterId={looked_up.get('matterId')!r} "
                      f"(the substitute's matterId is {substitute['matterId']!r})")
            continue
        print(f"--- actions of {record['recordNumber']}")
        display(action_table(record))

# Originals with no item file are looked up by record number (they may not exist as separate records).
to_fetch = sorted(set(to_fetch)) + [RECORD_NUMBER_LOOKUP_CONTROL]
print(f"\nControl lookup {RECORD_NUMBER_LOOKUP_CONTROL}: {lookup_result(RECORD_NUMBER_LOOKUP_CONTROL)}")
(CHECK_REQUESTS_DIR / "substitute_example_originals.json").write_text(json.dumps({
    "why": "originals of the section 19 substitute examples with no item file; looked up by record number",
    "record_numbers": to_fetch}, indent=1))
print(f"\n{len(to_fetch)} originals to look up by record number: {to_fetch}. Fetch with:")
print("  python scripts/fetch_elms.py matters_by_id data/raw/elms/check_requests/substitute_example_originals.json matters_substitute_examples")